In [ ]:
"""
SSL Pretext Tasks — Animal Classification Pipeline  (TPU-v3/v4/v5e friendly)
=============================================================================
Phase 1: Multi-task classical SSL pretraining on unlabeled images:
  - Rotation prediction  (0 / 90 / 180 / 270 deg)
  - Jigsaw puzzle        (3×3 grid → 100-perm subset of 9!)
  - Colorization         (grayscale L → predict ab channels in Lab space)

Phase 2: Transfer encoder → fine-tune on 1 000 labeled images (10 classes).

Final model : torchvision.models.resnet18(num_classes=10)  [no pretrained weights]
Submission  : id column has NO file extension  (e.g. "0", not "0.jpg")
"""

import json
from itertools import permutations

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.optim.lr_scheduler import CosineAnnealingLR, StepLR
from torch.utils.data import DataLoader, Dataset
from torchvision import models

# ── TPU / XLA setup ──────────────────────────────────────────────────────────
try:
    import torch_xla.core.xla_model as xm
    import torch_xla.distributed.parallel_loader as pl
    _ON_TPU = True
    DEVICE = xm.xla_device()
    print(f"[TPU] Using XLA device: {DEVICE}")
except Exception:
    _ON_TPU = False
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[CPU/GPU] torch_xla not found, falling back to: {DEVICE}")

def _step(optimizer: optim.Optimizer) -> None:
    if _ON_TPU:
        xm.optimizer_step(optimizer)
    else:
        optimizer.step()

def _mark() -> None:
    if _ON_TPU:
        xm.mark_step()

def _print(*args, **kwargs) -> None:
    if _ON_TPU:
        xm.master_print(*args, **kwargs)
    else:
        print(*args, **kwargs)

def _wrap_loader(loader: DataLoader):
    if _ON_TPU:
        return pl.MpDeviceLoader(loader, DEVICE)
    return loader

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_DIR = "/kaggle/input/ssl-pretext-tasks"
# If your Kaggle mount is /kaggle/input/competitions/ssl-pretext-tasks, uncomment below:
# DATA_DIR = "/kaggle/input/competitions/ssl-pretext-tasks"

JIGSAW_GRID = 3
JIGSAW_PERM_SET = 100
PATCH_SIZE = 72  # 3*72=216 static crop

# TPU-friendly defaults; lower for quick debugging
SSL_EPOCHS = 20
SSL_BATCH = 128
SSL_LR = 3e-3

WARMUP_EPOCHS = 3
FT_HEAD_LR = 1e-3
FT_HEAD_STEP = 2
FT_HEAD_GAMMA = 0.3

FT_EPOCHS = 30
FT_BATCH = 128
FT_LR = 5e-4
FT_LR_MIN = 1e-6
NUM_CLASSES = 10

def _build_perm_set(n_patches: int, n_perms: int, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    all_p = np.array(list(permutations(range(n_patches))))
    idx = rng.choice(len(all_p), size=n_perms, replace=False)
    return all_p[idx]

PERM_SET = _build_perm_set(JIGSAW_GRID ** 2, JIGSAW_PERM_SET)

train_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomCrop(224, padding=16),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

_patch_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def make_jigsaw(img_np: np.ndarray):
    from PIL import Image
    side = JIGSAW_GRID * PATCH_SIZE
    img = Image.fromarray(img_np).resize((224, 224))
    left = (224 - side) // 2
    img = img.crop((left, left, left + side, left + side))

    patches = []
    for r in range(JIGSAW_GRID):
        for c in range(JIGSAW_GRID):
            box = (c * PATCH_SIZE, r * PATCH_SIZE, (c + 1) * PATCH_SIZE, (r + 1) * PATCH_SIZE)
            patches.append(_patch_transform(img.crop(box)))

    perm_idx = np.random.randint(0, JIGSAW_PERM_SET)
    perm = PERM_SET[perm_idx]
    shuffled = torch.stack([patches[i] for i in perm])
    return shuffled, int(perm_idx)

def make_colorization(img_np: np.ndarray):
    from PIL import Image
    img = Image.fromarray(img_np).resize((224, 224)).convert("RGB")
    try:
        from skimage.color import rgb2lab
        lab = rgb2lab(np.array(img, dtype=np.uint8)).astype(np.float32)
        L = torch.from_numpy(lab[:, :, 0:1] / 100.0).permute(2, 0, 1)
        ab = torch.from_numpy((lab[:, :, 1:3] + 128) / 255.0).permute(2, 0, 1)
    except Exception:
        gray = np.array(img.convert("L"), dtype=np.float32) / 255.0
        rgb = np.array(img, dtype=np.float32) / 255.0
        L = torch.from_numpy(gray).unsqueeze(0)
        ab = torch.from_numpy(rgb[:, :, :2]).permute(2, 0, 1)
    return L, ab

class MultiTaskSSLDataset(Dataset):
    def __init__(self, images):
        self.images = images

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        rot_label = np.random.randint(0, 4)
        rot_tensor = train_transform(np.rot90(img, rot_label).copy())
        jig_patches, jig_label = make_jigsaw(img)
        gray, ab = make_colorization(img)
        return rot_tensor, rot_label, jig_patches, jig_label, gray, ab

class LabeledDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx]), int(self.labels[idx])

class TestDataset(Dataset):
    def __init__(self, images, transform):
        self.images = images
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        return self.transform(self.images[idx])

def load_data(data_dir):
    train_images = np.load(f"{data_dir}/train_images.npy")
    train_labels = np.load(f"{data_dir}/train_labels.npy")
    unlabeled_images = np.load(f"{data_dir}/unlabeled_images.npy")
    test_images = np.load(f"{data_dir}/test_images.npy")
    with open(f"{data_dir}/class_info.json") as f:
        class_info = json.load(f)
    idx_to_class = {i: name for i, name in enumerate(class_info["class_names"])}
    return train_images, train_labels, unlabeled_images, test_images, idx_to_class

class JigsawEncoder(nn.Module):
    def __init__(self, shared_encoder, feat_dim):
        super().__init__()
        self.encoder = shared_encoder
        self.head = nn.Sequential(
            nn.Linear(feat_dim * JIGSAW_GRID ** 2, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, JIGSAW_PERM_SET),
        )

    def forward(self, patches):
        B, N, C, H, W = patches.shape
        feats = self.encoder(patches.view(B * N, C, H, W)).flatten(1)
        return self.head(feats.view(B, N * feats.shape[-1]))

class ColorizationDecoder(nn.Module):
    def __init__(self, feat_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim, 512 * 7 * 7),
            nn.ReLU(inplace=True),
            nn.Unflatten(1, (512, 7, 7)),
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 2, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)

class GrayEncoder(nn.Module):
    def __init__(self, shared_encoder):
        super().__init__()
        self.encoder = shared_encoder
        old_conv = shared_encoder[0]
        self.gray_conv = nn.Conv2d(
            1, old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )
        with torch.no_grad():
            self.gray_conv.weight.copy_(old_conv.weight.mean(dim=1, keepdim=True))

    def forward(self, x):
        x = self.gray_conv(x)
        for layer in list(self.encoder.children())[1:]:
            x = layer(x)
        return x.flatten(1)

class MultiTaskSSLModel(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=None)
        feat_dim = base.fc.in_features
        self.encoder = nn.Sequential(*list(base.children())[:-1])
        self.rot_head = nn.Linear(feat_dim, 4)
        self.jigsaw_module = JigsawEncoder(self.encoder, feat_dim)
        self.gray_encoder = GrayEncoder(self.encoder)
        self.colorization_head = ColorizationDecoder(feat_dim)

    def forward_rotation(self, x):
        return self.rot_head(self.encoder(x).flatten(1))

    def forward_jigsaw(self, patches):
        return self.jigsaw_module(patches)

    def forward_colorization(self, gray):
        return self.colorization_head(self.gray_encoder(gray))

def build_classifier(ssl_model):
    resnet = models.resnet18(num_classes=NUM_CLASSES, weights=None)
    enc = ssl_model.encoder
    resnet.conv1 = enc[0]
    resnet.bn1 = enc[1]
    resnet.relu = enc[2]
    resnet.maxpool = enc[3]
    resnet.layer1 = enc[4]
    resnet.layer2 = enc[5]
    resnet.layer3 = enc[6]
    resnet.layer4 = enc[7]
    return resnet

def pretrain(unlabeled_images):
    loader = DataLoader(
        MultiTaskSSLDataset(unlabeled_images),
        batch_size=SSL_BATCH,
        shuffle=True,
        num_workers=0,
        drop_last=True,
    )
    model = MultiTaskSSLModel().to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=SSL_LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=SSL_EPOCHS, eta_min=1e-5)
    rot_criterion = nn.CrossEntropyLoss()
    jig_criterion = nn.CrossEntropyLoss()
    col_criterion = nn.MSELoss()

    for epoch in range(SSL_EPOCHS):
        model.train()
        total = rot_t = jig_t = col_t = 0.0
        for batch in _wrap_loader(loader):
            rot_img, rot_lbl, jig_patches, jig_lbl, gray, ab = batch
            if not _ON_TPU:
                rot_img = rot_img.to(DEVICE)
                rot_lbl = rot_lbl.to(DEVICE)
                jig_patches = jig_patches.to(DEVICE)
                jig_lbl = jig_lbl.to(DEVICE)
                gray = gray.to(DEVICE)
                ab = ab.to(DEVICE)

            optimizer.zero_grad()
            l_rot = rot_criterion(model.forward_rotation(rot_img), rot_lbl)
            l_jig = jig_criterion(model.forward_jigsaw(jig_patches), jig_lbl)
            l_col = col_criterion(model.forward_colorization(gray), ab)
            loss = l_rot + l_jig + l_col
            loss.backward()
            _step(optimizer)
            _mark()

            total += float(loss.item())
            rot_t += float(l_rot.item())
            jig_t += float(l_jig.item())
            col_t += float(l_col.item())

        scheduler.step()
        n = len(loader)
        _print(f"[SSL] {epoch+1}/{SSL_EPOCHS} total={total/n:.4f} rot={rot_t/n:.4f} jig={jig_t/n:.4f} col={col_t/n:.4f} lr={scheduler.get_last_lr()[0]:.2e}")

    return model

def finetune(model, train_images, train_labels):
    loader = DataLoader(
        LabeledDataset(train_images, train_labels, train_transform),
        batch_size=FT_BATCH,
        shuffle=True,
        num_workers=0,
        drop_last=True,
    )
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    for p in model.parameters():
        p.requires_grad = False
    for p in model.fc.parameters():
        p.requires_grad = True

    optimizer = optim.Adam(model.fc.parameters(), lr=FT_HEAD_LR)
    scheduler = StepLR(optimizer, step_size=FT_HEAD_STEP, gamma=FT_HEAD_GAMMA)

    for epoch in range(WARMUP_EPOCHS):
        model.train()
        total_loss = 0.0
        for imgs, labels in _wrap_loader(loader):
            if not _ON_TPU:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            _step(optimizer)
            _mark()
            total_loss += float(loss.item())
        scheduler.step()
        _print(f"[Warmup] {epoch+1}/{WARMUP_EPOCHS} loss={total_loss/len(loader):.4f} lr={scheduler.get_last_lr()[0]:.2e}")

    for p in model.parameters():
        p.requires_grad = True

    optimizer = optim.Adam(model.parameters(), lr=FT_LR, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=FT_EPOCHS, eta_min=FT_LR_MIN)

    for epoch in range(FT_EPOCHS):
        model.train()
        total_loss = 0.0
        for imgs, labels in _wrap_loader(loader):
            if not _ON_TPU:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            _step(optimizer)
            _mark()
            total_loss += float(loss.item())
        scheduler.step()
        _print(f"[Finetune] {epoch+1}/{FT_EPOCHS} loss={total_loss/len(loader):.4f} lr={scheduler.get_last_lr()[0]:.2e}")

    return model

def predict(model, test_images):
    loader = DataLoader(TestDataset(test_images, test_transform), batch_size=FT_BATCH, shuffle=False, num_workers=0, drop_last=False)
    model.eval()
    preds = []
    with torch.no_grad():
        for imgs in _wrap_loader(loader):
            if not _ON_TPU:
                imgs = imgs.to(DEVICE)
            out = model(imgs).argmax(dim=1)
            _mark()
            preds.extend(out.cpu().tolist())
    return preds

def main():
    _print(f"Device: {DEVICE}")
    _print(f"On TPU: {_ON_TPU}\n")

    _print("Loading data ...")
    train_images, train_labels, unlabeled_images, test_images, idx_to_class = load_data(DATA_DIR)

    _print("\n===== Phase 1: SSL Pretraining =====")
    ssl_model = pretrain(unlabeled_images)

    _print("\n===== Phase 2: Fine-tuning =====")
    classifier = build_classifier(ssl_model)
    classifier = finetune(classifier, train_images, train_labels)

    _print("\n===== Inference =====")
    preds = predict(classifier, test_images)

    submission = pd.DataFrame({
        "id": [str(i) for i in range(len(preds))],
        "class": [idx_to_class[p] for p in preds],
    })
    submission.to_csv("submission.csv", index=False)
    _print("Saved submission.csv")
    _print(submission.head(10).to_string(index=False))

main()
